In [4]:
from pathlib import Path
import numpy as np
from utils.read_write_model import read_model, qvec2rotmat

def pose_from_image(img):
    """返回 (R, t)"""
    R = qvec2rotmat(img.qvec)
    t = img.tvec
    return R, t

def compare_models(sparse_dir, tri_dir):
    # sparse 用 txt
    cams_s, imgs_s, _ = read_model(sparse_dir, ext=".txt")
    # triangulated 用 bin
    cams_t, imgs_t, _ = read_model(tri_dir, ext=".bin")

    # 按图像名对应
    common = set(i.name for i in imgs_s.values()) & set(i.name for i in imgs_t.values())
    print(f"共有 {len(common)} 张图片可比较")

    max_R_diff = 0
    max_t_diff = 0

    for name in sorted(common):
        s_img = [i for i in imgs_s.values() if i.name == name][0]
        t_img = [i for i in imgs_t.values() if i.name == name][0]

        R1, t1 = pose_from_image(s_img)
        R2, t2 = pose_from_image(t_img)

        R_diff = np.linalg.norm(R1 - R2)
        t_diff = np.linalg.norm(t1 - t2)

        max_R_diff = max(max_R_diff, R_diff)
        max_t_diff = max(max_t_diff, t_diff)

        if R_diff > 1e-6 or t_diff > 1e-6:
            print(f"[WARNING] {name}: R_diff={R_diff:.2e}, t_diff={t_diff:.2e}")

    print(f"\n最大旋转差: {max_R_diff:.2e}")
    print(f"最大平移差: {max_t_diff:.2e}")
    if max_R_diff < 1e-8 and max_t_diff < 1e-8:
        print("✅ 两个文件夹的位姿完全一致")
    else:
        print("⚠️ 存在数值差异（可能只是浮点精度）")

if __name__ == "__main__":
    sparse_dir = Path("/home/yuzhou/Desktop/datasets/nerf_synthetic_colmap/chair/colmap_results/sparse/0")
    tri_dir   = Path("/home/yuzhou/Desktop/datasets/nerf_synthetic_colmap/chair/colmap_results/triangulated")
    compare_models(sparse_dir, tri_dir)


共有 100 张图片可比较
[WARNING] r_10.png: R_diff=2.22e-06, t_diff=0.00e+00
[WARNING] r_11.png: R_diff=2.72e-06, t_diff=0.00e+00
[WARNING] r_13.png: R_diff=3.09e-06, t_diff=0.00e+00
[WARNING] r_14.png: R_diff=1.26e-06, t_diff=0.00e+00
[WARNING] r_16.png: R_diff=3.00e-06, t_diff=0.00e+00
[WARNING] r_17.png: R_diff=1.75e-06, t_diff=0.00e+00
[WARNING] r_18.png: R_diff=1.54e-06, t_diff=0.00e+00
[WARNING] r_19.png: R_diff=2.49e-06, t_diff=0.00e+00
[WARNING] r_20.png: R_diff=1.35e-06, t_diff=0.00e+00
[WARNING] r_22.png: R_diff=2.32e-06, t_diff=0.00e+00
[WARNING] r_27.png: R_diff=2.84e-06, t_diff=0.00e+00
[WARNING] r_28.png: R_diff=1.25e-06, t_diff=0.00e+00
[WARNING] r_29.png: R_diff=1.88e-06, t_diff=0.00e+00
[WARNING] r_3.png: R_diff=3.39e-06, t_diff=0.00e+00
[WARNING] r_32.png: R_diff=1.43e-06, t_diff=0.00e+00
[WARNING] r_33.png: R_diff=2.42e-06, t_diff=0.00e+00
[WARNING] r_36.png: R_diff=1.85e-06, t_diff=0.00e+00
[WARNING] r_39.png: R_diff=1.84e-06, t_diff=0.00e+00
[WARNING] r_4.png: R_diff=1.55e-0

In [6]:
from pathlib import Path
from utils.read_write_model import read_model

def count_points(model_dir, ext=".bin"):
    cams, imgs, points3D = read_model(model_dir, ext=ext)
    return len(points3D)

if __name__ == "__main__":
    sparse_dir = Path("/home/yuzhou/Desktop/datasets/nerf_synthetic_colmap/chair/colmap_results/sparse/0")
    tri_dir    = Path("/home/yuzhou/Desktop/datasets/nerf_synthetic_colmap/chair/colmap_results/triangulated")

    n_sparse = count_points(sparse_dir, ext=".txt")
    n_tri    = count_points(tri_dir, ext=".bin")

    print(f"Sparse点云数量:       {n_sparse}")
    print(f"Triangulated点云数量: {n_tri}")
    #print(f"增加了 {n_tri - n_sparse} 个点 (×{n_tri/n_sparse:.2f} 倍)")


Sparse点云数量:       0
Triangulated点云数量: 14121
